# 2 — Training-time defences

Two ways of making the model itself harder to attack, as opposed to
pre-processing its input:

1. **Data augmentation** — fine-tune on rotated, cropped, flipped and
   colour-jittered images. Nothing adversarial is involved.
2. **Adversarial training** — fine-tune on a mixture of clean, FGSM and PGD
   images, regenerated every batch against the current weights.

Both start from the same pretrained ResNet-20 and both are evaluated with the
same attacks at the same budget, so the two rows of the results table can be
read against each other and against notebook 1.

`advlab` defines every attack on images in `[0, 1]`, so `eps=8/255` is eight
levels of an 8-bit pixel — the standard CIFAR-10 L-infinity budget — and it
means the same thing in all four notebooks.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import advlab
from advlab import CLASSES, DEVICE, accuracy, apgd, cifar10, fgsm, mifgsm, pgd

advlab.set_seed()
print("device:", DEVICE)

In [ ]:
EPS = 8 / 255
BATCH = 128

test_loader = cifar10(train=False, batch_size=256)
# Half the training set, fixed by seed: enough to fine-tune, quick per epoch.
train_plain = cifar10(train=True, batch_size=BATCH, fraction=0.5)
train_augmented = cifar10(train=True, batch_size=BATCH, fraction=0.5,
                          augment=True)

baseline = advlab.resnet20(pretrained=True)
print(f"baseline clean: {accuracy(baseline, test_loader):.2f}%")

In [ ]:
def report(model, label):
    """Clean / FGSM / PGD accuracy at the shared budget."""
    a_clean = accuracy(model, test_loader)
    a_fgsm = accuracy(model, test_loader, attack=fgsm, eps=EPS)
    a_pgd = accuracy(model, test_loader, attack=pgd, eps=EPS,
                     alpha=2 / 255, steps=10)
    print(f"{label:24s} clean {a_clean:6.2f}%   FGSM {a_fgsm:6.2f}%"
          f"   PGD {a_pgd:6.2f}%")
    return a_clean, a_fgsm, a_pgd


def umap_of(model, title, **kw):
    feats, labels, conds = advlab.collect_features(
        model, test_loader, n_samples=1000, eps=EPS, **kw)
    advlab.plot_umap(feats, labels, conds, title)
    plt.show()

## Defence 1 — data augmentation

Ten epochs of fine-tuning at `lr = 1e-4`. The transforms are geometric and
photometric; none of them has any relationship to the gradient of the loss,
which is the point of the comparison.

In [ ]:
EPOCHS = 10

aug_model = advlab.resnet20(pretrained=True)
opt = torch.optim.Adam(aug_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

aug_history = []
for epoch in range(EPOCHS):
    aug_model.train()
    running = 0.0
    for images, labels in train_augmented:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(aug_model(images), labels)
        loss.backward()
        opt.step()
        running += loss.item()
    aug_history.append(running / len(train_augmented))
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss {aug_history[-1]:.4f}")

In [ ]:
aug_scores = report(aug_model, "data augmentation")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), aug_history, marker="o")
plt.xlabel("epoch"), plt.ylabel("cross-entropy")
plt.title("Data augmentation: training loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
umap_of(aug_model, "Penultimate features after augmentation fine-tuning")

## Defence 2 — adversarial training

Three epochs at `lr = 1e-4`, minimising the mean of three losses per batch:
clean, FGSM and PGD. The adversarial images are regenerated against the
current weights on every batch, which is what makes this expensive and also
what makes it work.

Two details that are easy to get wrong:

- Attacks are generated with the model in `eval()` mode and the weights are
  updated in `train()` mode. Generating them in `train()` mode would push the
  attack's forward passes through BatchNorm's running-statistics update, so
  the statistics would drift towards the adversarial distribution as a side
  effect of attacking.
- `opt.zero_grad()` comes *after* the attacks are generated. Each attack
  backpropagates to its input, which also deposits gradients on the
  parameters; zeroing first and then attacking would add that attack gradient
  to the update.

In [ ]:
EPOCHS_ADV = 3

adv_model = advlab.resnet20(pretrained=True)
opt = torch.optim.Adam(adv_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

adv_history = []
for epoch in range(EPOCHS_ADV):
    running = 0.0
    for images, labels in train_plain:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # eval() mode: no BatchNorm statistics update while attacking.
        adv_model.eval()
        x_fgsm = fgsm(adv_model, images, labels, eps=EPS)
        x_pgd = pgd(adv_model, images, labels, eps=EPS, alpha=2 / 255, steps=10)

        adv_model.train()
        opt.zero_grad()          # after the attacks, not before
        loss = (loss_fn(adv_model(images), labels)
                + loss_fn(adv_model(x_fgsm), labels)
                + loss_fn(adv_model(x_pgd), labels)) / 3
        loss.backward()
        opt.step()
        running += loss.item()

    adv_history.append(running / len(train_plain))
    print(f"epoch {epoch + 1}/{EPOCHS_ADV}  loss {adv_history[-1]:.4f}")

In [ ]:
adv_scores = report(adv_model, "adversarial training")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS_ADV + 1), adv_history, marker="o")
plt.xlabel("epoch"), plt.ylabel("mean of clean / FGSM / PGD loss")
plt.title("Adversarial training: training loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
umap_of(adv_model, "Penultimate features after adversarial training")

## Comparison

In [ ]:
base_scores = report(baseline, "no defence")
print()
print(f"{'defence':24s} {'clean':>8s} {'FGSM':>8s} {'PGD':>8s}")
for label, s in [("no defence", base_scores),
                 ("data augmentation", aug_scores),
                 ("adversarial training", adv_scores)]:
    print(f"{label:24s} {s[0]:7.2f}% {s[1]:7.2f}% {s[2]:7.2f}%")

**Data augmentation** buys a little robustness and costs clean accuracy. The
transforms it applies are not the perturbations the attacks make: rotating and
cropping teaches invariance to changes a camera might produce, while FGSM and
PGD move every pixel by a tiny amount in a direction chosen by the model's own
gradient. There is no reason invariance to one should imply invariance to the
other, and the numbers say it does not. What augmentation is genuinely good at
is reducing overfitting.

**Adversarial training** is the only defence here that trains on the thing it
is defending against, and it is much the strongest — it keeps most of the clean
accuracy while raising attacked accuracy substantially. The costs are real: a
forward-backward pass per attack step per batch, so roughly an order of
magnitude more compute per epoch, and a tendency towards robust overfitting —
robustness to the attack used in training that does not transfer to a
different or stronger one. Checking it against APGD rather than the PGD it
trained on is the honest test.